# This notebook is broken down into 2 sections:
## 1. Data preprocessing - to process the data used for illustration into equal samples using propensity scoring. Using the PC algorithm we show that our treatment variable does not have any confounding effect
## 2. Run our RCT study analysis based on the treatment and the control groups

# 1. Data preprocessing

In [ ]:
import pandas as pd
import os

path = os.getcwd()
file_path = path + '\\data\student_performance_data.csv'
data_sp = pd.read_csv(file_path, header=0,sep=',')
data_sp.info()

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, mutual_info_regression

# Separate features and target
X = data_sp[
    ['Age', 'Gender', 'Ethnicity', 'ParentalEducation', 'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport', 
     'Extracurricular', 'Sports', 'Music', 'Volunteering']]
y = data_sp['GradeClass']

# Define a wrapper function to inject random_state into mutual_info_regression
def mi_regression_with_seed(X, y):
    return mutual_info_regression(X, y, random_state=23), None

# Using SelectKBest to select the top features based on mutual information
selector = SelectKBest(score_func=mi_regression_with_seed, k='all')
selector.fit(X, y)

# Get the scores and the p-values (if available, p-values might not be available for certain metrics like mutual_info_regression)
scores = selector.scores_

# Create a DataFrame to view the scores
feature_scores = pd.DataFrame({'Feature': X.columns, 'Score': scores})
feature_scores = feature_scores.sort_values(by='Score', ascending=False)

print(feature_scores)
relevant_scores = []
for item in feature_scores.iterrows():
    if (item[1]['Score'] > 0.0):
        relevant_scores.append(item[1]['Feature'])

print(relevant_scores)
relevant_scores


In [85]:
new_education_program = 'Tutoring'

In [ ]:

print(f'Min {new_education_program}:', data_sp[new_education_program].min())
print(f'Max {new_education_program}:', data_sp[new_education_program].max())

data_sp[new_education_program].plot(kind='hist', title=new_education_program)

**Target Variable: Grade Class**

GradeClass: Classification of students' grades based on GPA:

| GradeClass | Grade | GPA |
| ---------- | ----- | --- |
| 0 | A | (GPA >= 3.5) |
| 1 | B | (3.0 <= GPA < 3.5) |
| 2 | C | (2.5 <= GPA < 3.0) |
| 3 | D | (2.0 <= GPA < 2.5) |
| 4 | F | (GPA < 2.0) |

We define high performance to only be the grade category 'A'

In [ ]:
relevant_all = relevant_scores.copy()
relevant_all.append("GradeClass")

df = data_sp[relevant_all].copy()

# Define high performance as only A
df['High_Performance'] = df['GradeClass'].apply(lambda x: 1 if x < 1 else 0)

print("Amount of samples in treatment group: ", df[df['High_Performance']==1].shape[0])
print("Amount of samples in control group: ", df[df['High_Performance']==0].shape[0])

In [ ]:
covariates = list(set(relevant_scores) - set([new_education_program])) # the treatment variable should not be included in the covariates for the propensity score estimation!
print(covariates)

Before using propensity score matching we show that our treatment variable does not have any confounding effect.

In [ ]:
from causallearn.search.ConstraintBased.PC import pc

data_all = data_sp[relevant_all]

# AUTOMATICALLY ESTIMATED GRAPH FROM DATA!
labels = [f'{col}' for i, col in enumerate(data_all.columns)]
data = data_all.to_numpy()

# Alpha level is 0.05 since I am interested in a causal representation as close as possible to the truth
# A p-value of 0.05 indicates that we cannot reject the null hypothesis that the causal relationship depicted in the graph is correct
cg = pc(data, alpha=0.05)
cg.draw_pydot_graph(labels=labels)

Now we use propensity score matching to derive groups of equal samples.

In [ ]:
# Propensity score matching to obtain quasi-RCT
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

features = df[covariates]
treatment = df[new_education_program]

# Use scaled features and logistic regression for propensity score prediction!
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, solver='lbfgs'))

# Fit the model
model.fit(features, treatment)

In [91]:
df['propensity_score'] = model.predict_proba(features)[:, 1]

control = df[df[new_education_program] == 0].sort_values('propensity_score')
treated = df[df[new_education_program] == 1].sort_values('propensity_score')

In [ ]:

print(control['propensity_score'])
print(treated.shape)

In [93]:
# A small method to perform matching fast, i.e. in complexity O(N + M)
def perform_matching(treated_df, control_df, tolerance=0.01):
    # NOTE: Assumes treated_df and control_df to be sorted in ascending order with respect to the column "propensity_score"
    treated_matched_indices = []  # To store indices of matched treated samples
    control_matched_indices = []  # To store indices of matched control samples
    
    i, j = 0, 0  # Initialize pointers
    
    # Extract the propensity scores assuming they are in a column named 'propensity_score'
    treated_scores = treated_df['propensity_score'].values
    control_scores = control_df['propensity_score'].values
    
    while i < len(treated_scores) and j < len(control_scores):
        if abs(treated_scores[i] - control_scores[j]) <= tolerance:
            # If within tolerance, consider it a match
            treated_matched_indices.append(i)
            control_matched_indices.append(j)
            # Move both pointers to find next match
            i += 1
            j += 1
        elif treated_scores[i] < control_scores[j]:
            # Increment i to get closer to j
            i += 1
        else:
            # Increment j to get closer to i
            j += 1
    
    # Create new dataframes for matched samples
    treated_matched = treated_df.iloc[treated_matched_indices]
    control_matched = control_df.iloc[control_matched_indices]
    
    return treated_matched, control_matched

# Assume treated and control are your dataframes
treated_matched, control_matched = perform_matching(treated, control)

In [ ]:
print("Amount of samples in treatment group: ", treated_matched.shape[0])
print("Amount of samples in control group: ", control_matched.shape[0])

# Save matched data to csv for quasi RCT study -> see part 2. RCT Study
treated_matched.to_csv('treated_matched.csv', index=False)
control_matched.to_csv('control_matched.csv', index=False)

We now have equal samples in both the treatment and the control group

In [ ]:
treated_matched['GradeClass'].plot(kind='hist', title='GradeClass - treated')

In [ ]:
control_matched['GradeClass'].plot(kind='hist', title='GradeClass - control')

# 2. RCT Study

In [97]:
# Load the data, this is actually quasi RCT data, preprocessed above
treated_RCT = pd.read_csv("treated_matched.csv")
control_RCT = pd.read_csv("control_matched.csv")

For our required sample size calculation we use:
- standard benchmark effect size (w) of 0.3 (medium)
- significance level (α) set commonly to 0.05
- power (1 - β) set typically to 0.8

In [ ]:
from scipy.stats import norm

# Significance level
alpha = 0.05
beta = 0.2
power = 1 - beta
w = 0.3
# Calculate the Z_alpha value
Z_alpha2 = norm.ppf(1 - alpha/2)
Z_beta = norm.ppf(power)

N = (2 * (Z_alpha2 + Z_beta)**2)/w**2

print(f"The minimal sample number per class is: ", N)

In [ ]:

print("Amount of samples in treatment group: ", treated_RCT.shape[0])
print("Amount of samples in control group: ", control_RCT.shape[0])

Y_1 = treated_RCT['High_Performance'].apply(lambda x: 1 if x >= 1 else 0).mean()
print("The average of higher performance in the treatment group is: ", Y_1)
Y_0 = control_RCT['High_Performance'].apply(lambda x: 1 if x >= 1 else 0).mean()
print("The average of higher performance in the control group is: ", Y_0)

print("The average treatment effect (ATE) is: ", Y_1 - Y_0)

Our required sample size based on Cohen's w 0.3 is 175 per group. We have 721 samples per group which is more than sufficient. We can now run our 2 sided chi square hypothesis test of independence.

Our hypothesis:
- Null Hypothesis (H₀): The intervention tutoring has no effect on the outcome gradeclass. The distribution of productivity improvement is the same for both groups.
- Alternative Hypothesis (Hₐ): The intervention tutoring has an effect on the outcome gradeclass. The distribution of productivity improvement differs between the groups.


In [ ]:
# Perform the chi-square test on the matched sample to assess the impact of tutoring on student performance.
matched_dataset = pd.concat([control_RCT, treated_RCT], axis=0)

from scipy.stats import chi2_contingency
import scipy.stats as stats

# Contingency table
contingency_table = pd.crosstab(matched_dataset[new_education_program], matched_dataset['High_Performance'])

# Chi-square test
chi2, p, dof, ex = chi2_contingency(contingency_table)

print("Degrees of freedom: ", dof)
print(f'Chi-square statistic: {chi2}, p-value: {p}')

# Calculate the critical value using the percent point function (inverse of cdf)
critical_value = stats.chi2.ppf((1 - alpha/2), dof)

print(f"Critical value at {alpha/2} significance level: {critical_value}")

if (chi2 > critical_value):
    print(f"Reject H₀ with p={p} -> that means the causal effect of tutoring on gradeclass is significant with alpha level {alpha/2}.")
else:
    print("Cannot reject H₀!")

**Conclusion**: The causal effect of tutoring on gradeclass (i.e. the performance of a student) is statistically significant!